In [1]:
# ==========================================
# AI STARTUP VALIDATOR
# ML Experimentation
# ==========================================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

from xgboost import XGBClassifier

from sklearn.pipeline import Pipeline
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import warnings
warnings.filterwarnings("ignore")

In [2]:
import os

folder_path = r"C:\Users\shubham kharb\Desktop\Minor Project\Datasets"

print(os.listdir(folder_path))

['startup_failure_analysis.csv', 'startup_profiles.csv', 'startup_success_prediction.csv']


In [3]:
file_path = r"C:\Users\shubham kharb\Desktop\Minor Project\Datasets\startup_success_prediction.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

Dataset loaded successfully!
Shape: (923, 49)


In [4]:
df.head()

,Unnamed: 0,state_code,latitude,longitude,zip_code,id,city,Unnamed: 6,name,labels,...,object_id,has_VC,has_angel,has_roundA,has_roundB,has_roundC,has_roundD,avg_participants,is_top500,status
0,1005,CA,42.358880,-71.056820,92101,c:6669,San Diego,NaN,Bandsintown,1,...,c:6669,0,1,0,0,0,0,1.0000,0,acquired
1,204,CA,37.238916,-121.973718,95032,c:16283,Los Gatos,NaN,TriCipher,1,...,c:16283,1,0,0,1,1,1,4.7500,1,acquired
2,1001,CA,32.901049,-117.192656,92121,c:65620,San Diego,San Diego CA 92121,Plixi,1,...,c:65620,0,0,1,0,0,0,4.0000,1,acquired
3,738,CA,37.320309,-122.050040,95014,c:42668,Cupertino,Cupertino CA 95014,Solidcore Systems,1,...,c:42668,0,0,0,1,1,1,3.3333,1,acquired
4,1002,CA,37.779281,-122.419236,94105,c:65806,San Francisco,San Francisco CA 94105,Inhale Digital,0,...,c:65806,1,1,0,0,0,0,1.0000,1,closed


In [5]:
df.columns.tolist()

['Unnamed: 0',
 'state_code',
 'latitude',
 'longitude',
 'zip_code',
 'id',
 'city',
 'Unnamed: 6',
 'name',
 'labels',
 'founded_at',
 'closed_at',
 'first_funding_at',
 'last_funding_at',
 'age_first_funding_year',
 'age_last_funding_year',
 'age_first_milestone_year',
 'age_last_milestone_year',
 'relationships',
 'funding_rounds',
 'funding_total_usd',
 'milestones',
 'state_code.1',
 'is_CA',
 'is_NY',
 'is_MA',
 'is_TX',
 'is_otherstate',
 'category_code',
 'is_software',
 'is_web',
 'is_mobile',
 'is_enterprise',
 'is_advertising',
 'is_gamesvideo',
 'is_ecommerce',
 'is_biotech',
 'is_consulting',
 'is_othercategory',
 'object_id',
 'has_VC',
 'has_angel',
 'has_roundA',
 'has_roundB',
 'has_roundC',
 'has_roundD',
 'avg_participants',
 'is_top500',
 'status']

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 923 entries, 0 to 922
Data columns (total 49 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Unnamed: 0                923 non-null    int64  
 1   state_code                923 non-null    object 
 2   latitude                  923 non-null    float64
 3   longitude                 923 non-null    float64
 4   zip_code                  923 non-null    object 
 5   id                        923 non-null    object 
 6   city                      923 non-null    object 
 7   Unnamed: 6                430 non-null    object 
 8   name                      923 non-null    object 
 9   labels                    923 non-null    int64  
 10  founded_at                923 non-null    object 
 11  closed_at                 335 non-null    object 
 12  first_funding_at          923 non-null    object 
 13  last_funding_at           923 non-null    object 
 14  age_first_

In [7]:
df["status"].value_counts()

status
acquired    597
closed      326
Name: count, dtype: int64

In [8]:
df.isnull().sum()

Unnamed: 0                    0
state_code                    0
latitude                      0
longitude                     0
zip_code                      0
id                            0
city                          0
Unnamed: 6                  493
name                          0
labels                        0
founded_at                    0
closed_at                   588
first_funding_at              0
last_funding_at               0
age_first_funding_year        0
age_last_funding_year         0
age_first_milestone_year    152
age_last_milestone_year     152
relationships                 0
funding_rounds                0
funding_total_usd             0
milestones                    0
state_code.1                  1
is_CA                         0
is_NY                         0
is_MA                         0
is_TX                         0
is_otherstate                 0
category_code                 0
is_software                   0
is_web                        0
is_mobil

In [9]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


In [10]:
print("Target distribution:")
print(df["status"].value_counts())

print("\nTarget percentage:")
print(df["status"].value_counts(normalize=True) * 100)

Target distribution:
status
acquired    597
closed      326
Name: count, dtype: int64

Target percentage:
status
acquired    64.68039
closed      35.31961
Name: proportion, dtype: float64


In [11]:
df["status"].unique()

array(['acquired', 'closed'], dtype=object)

In [12]:
columns_to_check = [
    "state_code",
    "state_code.1",
    "category_code",
    "labels",
    "founded_at",
    "closed_at",
    "first_funding_at",
    "last_funding_at"
]

df[columns_to_check].head(10)

,state_code,state_code.1,category_code,labels,founded_at,closed_at,first_funding_at,last_funding_at
0,CA,CA,music,1,1/1/2007,NaN,4/1/2009,1/1/2010
1,CA,CA,enterprise,1,1/1/2000,NaN,2/14/2005,12/28/2009
2,CA,CA,web,1,3/18/2009,NaN,3/30/2010,3/30/2010
3,CA,CA,software,1,1/1/2002,NaN,2/17/2005,4/25/2007
4,CA,CA,games_video,0,8/1/2010,10/1/2012,8/1/2010,4/1/2012
5,CA,CA,network_hosting,0,1/1/2002,2/15/2009,7/18/2006,7/18/2006
6,CA,CA,software,1,1/1/2005,NaN,9/21/2006,3/18/2010
7,CA,CA,finance,1,1/1/2004,NaN,8/24/2005,10/4/2010
8,MA,MA,mobile,1,1/1/2002,NaN,8/2/2005,2/8/2013
9,CA,CA,web,1,6/1/2005,NaN,2/1/2007,2/5/2010


In [13]:
for col in columns_to_check:
    print(f"\n========== {col} ==========")
    print(df[col].value_counts().head(15))


========== state_code ==========
state_code
CA    488
NY    106
MA     83
TX     42
WA     42
CO     19
IL     18
PA     17
VA     13
GA     11
NC      7
OR      7
NJ      7
MD      7
FL      6
Name: count, dtype: int64

========== state_code.1 ==========
state_code.1
CA    487
NY    106
MA     83
TX     42
WA     42
CO     19
IL     18
PA     17
VA     13
GA     11
NC      7
OR      7
NJ      7
MD      7
FL      6
Name: count, dtype: int64

========== category_code ==========
category_code
software            153
web                 144
mobile               79
enterprise           73
advertising          62
games_video          52
semiconductor        35
network_hosting      34
biotech              34
hardware             27
ecommerce            25
public_relations     25
cleantech            23
security             19
analytics            19
Name: count, dtype: int64

========== labels ==========
labels
1    597
0    326
Name: count, dtype: int64

========== founded_at ==========
fo

In [14]:
# ==========================================
# CREATE CLEAN DATASET
# ==========================================

df_clean = df.copy()

columns_to_remove = [
    "Unnamed: 0",
    "Unnamed: 6",
    "id",
    "object_id",
    "status",
    "closed_at",
    "state_code.1",
    "name",
    "city",
    "zip_code"
]

df_clean = df_clean.drop(
    columns=columns_to_remove,
    errors="ignore"
)

print("Original shape:", df.shape)
print("Cleaned shape:", df_clean.shape)

print("\nRemaining columns:")
print(df_clean.columns.tolist())

Original shape: (923, 49)
Cleaned shape: (923, 39)

Remaining columns:
['state_code', 'latitude', 'longitude', 'labels', 'founded_at', 'first_funding_at', 'last_funding_at', 'age_first_funding_year', 'age_last_funding_year', 'age_first_milestone_year', 'age_last_milestone_year', 'relationships', 'funding_rounds', 'funding_total_usd', 'milestones', 'is_CA', 'is_NY', 'is_MA', 'is_TX', 'is_otherstate', 'category_code', 'is_software', 'is_web', 'is_mobile', 'is_enterprise', 'is_advertising', 'is_gamesvideo', 'is_ecommerce', 'is_biotech', 'is_consulting', 'is_othercategory', 'has_VC', 'has_angel', 'has_roundA', 'has_roundB', 'has_roundC', 'has_roundD', 'avg_participants', 'is_top500']


In [15]:
df_clean.head()

,state_code,latitude,longitude,labels,founded_at,first_funding_at,last_funding_at,age_first_funding_year,age_last_funding_year,age_first_milestone_year,...,is_consulting,is_othercategory,has_VC,has_angel,has_roundA,has_roundB,has_roundC,has_roundD,avg_participants,is_top500
0,CA,42.358880,-71.056820,1,1/1/2007,4/1/2009,1/1/2010,2.2493,3.0027,4.6685,...,0,1,0,1,0,0,0,0,1.0000,0
1,CA,37.238916,-121.973718,1,1/1/2000,2/14/2005,12/28/2009,5.1260,9.9973,7.0055,...,0,0,1,0,0,1,1,1,4.7500,1
2,CA,32.901049,-117.192656,1,3/18/2009,3/30/2010,3/30/2010,1.0329,1.0329,1.4575,...,0,0,0,0,1,0,0,0,4.0000,1
3,CA,37.320309,-122.050040,1,1/1/2002,2/17/2005,4/25/2007,3.1315,5.3151,6.0027,...,0,0,0,0,0,1,1,1,3.3333,1
4,CA,37.779281,-122.419236,0,8/1/2010,8/1/2010,4/1/2012,0.0000,1.6685,0.0384,...,0,0,1,1,0,0,0,0,1.0000,1


In [16]:
df_clean.dtypes

state_code                   object
latitude                    float64
longitude                   float64
labels                        int64
founded_at                   object
first_funding_at             object
last_funding_at              object
age_first_funding_year      float64
age_last_funding_year       float64
age_first_milestone_year    float64
age_last_milestone_year     float64
relationships                 int64
funding_rounds                int64
funding_total_usd             int64
milestones                    int64
is_CA                         int64
is_NY                         int64
is_MA                         int64
is_TX                         int64
is_otherstate                 int64
category_code                object
is_software                   int64
is_web                        int64
is_mobile                     int64
is_enterprise                 int64
is_advertising                int64
is_gamesvideo                 int64
is_ecommerce                

In [17]:
# ==========================================
# DATE CONVERSION
# ==========================================

date_columns = [
    "founded_at",
    "first_funding_at",
    "last_funding_at"
]

for col in date_columns:
    df_clean[col] = pd.to_datetime(
        df_clean[col],
        errors="coerce"
    )

df_clean[date_columns].dtypes

founded_at          datetime64[ns]
first_funding_at    datetime64[ns]
last_funding_at     datetime64[ns]
dtype: object

In [18]:
# ==========================================
# DATE FEATURE ENGINEERING
# ==========================================

df_clean["founded_year"] = df_clean["founded_at"].dt.year
df_clean["first_funding_year"] = df_clean["first_funding_at"].dt.year
df_clean["last_funding_year"] = df_clean["last_funding_at"].dt.year

In [19]:
df_clean["years_to_first_funding"] = (
    df_clean["first_funding_year"] -
    df_clean["founded_year"]
)

df_clean["funding_period_years"] = (
    df_clean["last_funding_year"] -
    df_clean["first_funding_year"]
)

In [20]:
df_clean[
    [
        "founded_year",
        "first_funding_year",
        "last_funding_year",
        "years_to_first_funding",
        "funding_period_years"
    ]
].head(10)

,founded_year,first_funding_year,last_funding_year,years_to_first_funding,funding_period_years
0,2007,2009,2010,2,1
1,2000,2005,2009,5,4
2,2009,2010,2010,1,0
3,2002,2005,2007,3,2
4,2010,2010,2012,0,2
5,2002,2006,2006,4,0
6,2005,2006,2010,1,4
7,2004,2005,2010,1,5
8,2002,2005,2013,3,8
9,2005,2007,2010,2,3


In [21]:
df_clean = df_clean.drop(
    columns=[
        "founded_at",
        "first_funding_at",
        "last_funding_at"
    ]
)

In [22]:
state_indicator_columns = [
    "is_CA",
    "is_NY",
    "is_MA",
    "is_TX",
    "is_otherstate"
]

df_clean = df_clean.drop(
    columns=state_indicator_columns
)

In [23]:
category_indicator_columns = [
    "is_software",
    "is_web",
    "is_mobile",
    "is_enterprise",
    "is_advertising",
    "is_gamesvideo",
    "is_ecommerce",
    "is_biotech",
    "is_consulting",
    "is_othercategory"
]

df_clean = df_clean.drop(
    columns=category_indicator_columns
)

In [24]:
print("Shape:", df_clean.shape)

print("\nColumns:")
print(df_clean.columns.tolist())

print("\nData types:")
print(df_clean.dtypes)

Shape: (923, 26)

Columns:
['state_code', 'latitude', 'longitude', 'labels', 'age_first_funding_year', 'age_last_funding_year', 'age_first_milestone_year', 'age_last_milestone_year', 'relationships', 'funding_rounds', 'funding_total_usd', 'milestones', 'category_code', 'has_VC', 'has_angel', 'has_roundA', 'has_roundB', 'has_roundC', 'has_roundD', 'avg_participants', 'is_top500', 'founded_year', 'first_funding_year', 'last_funding_year', 'years_to_first_funding', 'funding_period_years']

Data types:
state_code                   object
latitude                    float64
longitude                   float64
labels                        int64
age_first_funding_year      float64
age_last_funding_year       float64
age_first_milestone_year    float64
age_last_milestone_year     float64
relationships                 int64
funding_rounds                int64
funding_total_usd             int64
milestones                    int64
category_code                object
has_VC                      

In [25]:
# ==========================================
# SEPARATE FEATURES AND TARGET
# ==========================================

X = df_clean.drop(columns=["labels"])
y = df_clean["labels"]

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

X shape: (923, 25)
y shape: (923,)

Target distribution:
labels
1    597
0    326
Name: count, dtype: int64


In [26]:
# ==========================================
# IDENTIFY FEATURE TYPES
# ==========================================

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

numerical_features = X.select_dtypes(
    exclude=["object"]
).columns.tolist()

print("Categorical features:")
print(categorical_features)

print("\nNumerical features:")
print(numerical_features)

Categorical features:
['state_code', 'category_code']

Numerical features:
['latitude', 'longitude', 'age_first_funding_year', 'age_last_funding_year', 'age_first_milestone_year', 'age_last_milestone_year', 'relationships', 'funding_rounds', 'funding_total_usd', 'milestones', 'has_VC', 'has_angel', 'has_roundA', 'has_roundB', 'has_roundC', 'has_roundD', 'avg_participants', 'is_top500', 'founded_year', 'first_funding_year', 'last_funding_year', 'years_to_first_funding', 'funding_period_years']


In [27]:
# ==========================================
# TRAIN / TEST SPLIT
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())


Training data: (738, 25)
Testing data: (185, 25)

Training target distribution:
labels
1    477
0    261
Name: count, dtype: int64

Testing target distribution:
labels
1    120
0     65
Name: count, dtype: int64


In [28]:
# ==========================================
# PREPROCESSING PIPELINES
# ==========================================


numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])

In [29]:
# ==========================================
# APPLY PREPROCESSING
# ==========================================

X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

Processed training shape: (738, 91)
Processed testing shape: (185, 91)


In [30]:
# ==========================================
# TRAIN MACHINE LEARNING MODELS
# ==========================================

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ),

    "XGBoost": XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric="logloss"
    )
}

In [31]:
# ==========================================
# MODEL TRAINING
# ==========================================

trained_models = {}

for name, model in models.items():

    print(f"Training {name}...")

    model.fit(
        X_train_processed,
        y_train
    )

    trained_models[name] = model

print("\nAll models trained successfully!")

Training Logistic Regression...
Training Random Forest...
Training Gradient Boosting...
Training XGBoost...

All models trained successfully!


In [32]:
# ==========================================
# MODEL EVALUATION
# ==========================================

results = []

for name, model in trained_models.items():

    y_pred = model.predict(X_test_processed)

    y_prob = model.predict_proba(
        X_test_processed
    )[:, 1]

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_prob)

    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "ROC-AUC": roc_auc
    })

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="F1 Score",
    ascending=False
)

results_df

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
1,Random Forest,0.805405,0.818182,0.900000,0.857143,0.837372
3,XGBoost,0.800000,0.832000,0.866667,0.848980,0.849231
2,Gradient Boosting,0.789189,0.813953,0.875000,0.843373,0.857179
0,Logistic Regression,0.751351,0.813559,0.800000,0.806723,0.797308


In [33]:
# ==========================================
# DETAILED CLASSIFICATION REPORT
# ==========================================

for name, model in trained_models.items():

    y_pred = model.predict(X_test_processed)

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    print(
        classification_report(
            y_test,
            y_pred,
            target_names=["Closed", "Acquired"]
        )
    )


Logistic Regression
              precision    recall  f1-score   support

      Closed       0.64      0.66      0.65        65
    Acquired       0.81      0.80      0.81       120

    accuracy                           0.75       185
   macro avg       0.73      0.73      0.73       185
weighted avg       0.75      0.75      0.75       185


Random Forest
              precision    recall  f1-score   support

      Closed       0.77      0.63      0.69        65
    Acquired       0.82      0.90      0.86       120

    accuracy                           0.81       185
   macro avg       0.80      0.77      0.78       185
weighted avg       0.80      0.81      0.80       185


Gradient Boosting
              precision    recall  f1-score   support

      Closed       0.73      0.63      0.68        65
    Acquired       0.81      0.88      0.84       120

    accuracy                           0.79       185
   macro avg       0.77      0.75      0.76       185
weighted avg       

In [34]:
results_df

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
1,Random Forest,0.805405,0.818182,0.900000,0.857143,0.837372
3,XGBoost,0.800000,0.832000,0.866667,0.848980,0.849231
2,Gradient Boosting,0.789189,0.813953,0.875000,0.843373,0.857179
0,Logistic Regression,0.751351,0.813559,0.800000,0.806723,0.797308


In [35]:
# ==========================================
# RANDOM FOREST - CLASSIFICATION REPORT
# ==========================================

rf_model = trained_models["Random Forest"]

y_pred_rf = rf_model.predict(X_test_processed)

print(classification_report(
    y_test,
    y_pred_rf,
    target_names=["Closed", "Acquired"]
))

              precision    recall  f1-score   support

      Closed       0.77      0.63      0.69        65
    Acquired       0.82      0.90      0.86       120

    accuracy                           0.81       185
   macro avg       0.80      0.77      0.78       185
weighted avg       0.80      0.81      0.80       185



In [36]:
# ==========================================
# RANDOM FOREST - CONFUSION MATRIX
# ==========================================

cm = confusion_matrix(y_test, y_pred_rf)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[ 41  24]
 [ 12 108]]


In [37]:
# ==========================================
# RANDOM FOREST HYPERPARAMETER TUNING
# ==========================================

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

param_grid_rf = {
    "n_estimators": [200, 300, 500],
    "max_depth": [None, 5, 10, 15],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}

grid_rf = GridSearchCV(
    estimator=rf,
    param_grid=param_grid_rf,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    verbose=1
)

grid_rf.fit(X_train_processed, y_train)

print("\nBest Parameters:")
print(grid_rf.best_params_)

print("\nBest Cross-Validation F1 Score:")
print(grid_rf.best_score_)

Fitting 5 folds for each of 216 candidates, totalling 1080 fits

Best Parameters:
{'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 500}

Best Cross-Validation F1 Score:
0.8611132830398344


In [38]:
# ==========================================
# EVALUATE TUNED RANDOM FOREST
# ==========================================

best_rf = grid_rf.best_estimator_

y_pred_best_rf = best_rf.predict(X_test_processed)

print("Classification Report:")
print(classification_report(
    y_test,
    y_pred_best_rf,
    target_names=["Closed", "Acquired"]
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_best_rf))

Classification Report:
              precision    recall  f1-score   support

      Closed       0.80      0.62      0.70        65
    Acquired       0.81      0.92      0.86       120

    accuracy                           0.81       185
   macro avg       0.81      0.77      0.78       185
weighted avg       0.81      0.81      0.80       185


Confusion Matrix:
[[ 40  25]
 [ 10 110]]


In [39]:
# ==========================================
# TUNED RANDOM FOREST METRICS
# ==========================================

y_prob_best_rf = best_rf.predict_proba(
    X_test_processed
)[:, 1]

print("Accuracy :", accuracy_score(y_test, y_pred_best_rf))
print("Precision:", precision_score(y_test, y_pred_best_rf))
print("Recall   :", recall_score(y_test, y_pred_best_rf))
print("F1 Score :", f1_score(y_test, y_pred_best_rf))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob_best_rf))

Accuracy : 0.8108108108108109
Precision: 0.8148148148148148
Recall   : 0.9166666666666666
F1 Score : 0.8627450980392157
ROC-AUC  : 0.8510256410256409


In [40]:
# ==========================================
# XGBOOST HYPERPARAMETER TUNING
# ==========================================

from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV

xgb = XGBClassifier(
    random_state=42,
    eval_metric="logloss"
)

param_grid_xgb = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.03, 0.05, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

grid_xgb = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid_xgb,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    verbose=1
)

grid_xgb.fit(X_train_processed, y_train)

print("\nBest Parameters:")
print(grid_xgb.best_params_)

print("\nBest Cross-Validation F1 Score:")
print(grid_xgb.best_score_)

Fitting 5 folds for each of 108 candidates, totalling 540 fits

Best Parameters:
{'colsample_bytree': 1.0, 'learning_rate': 0.05, 'max_depth': 5, 'n_estimators': 100, 'subsample': 1.0}

Best Cross-Validation F1 Score:
0.8590928369450392


In [41]:
# ==========================================
# EVALUATE TUNED XGBOOST
# ==========================================

best_xgb = grid_xgb.best_estimator_

y_pred_xgb = best_xgb.predict(X_test_processed)

y_prob_xgb = best_xgb.predict_proba(
    X_test_processed
)[:, 1]

print("Accuracy :", accuracy_score(y_test, y_pred_xgb))
print("Precision:", precision_score(y_test, y_pred_xgb))
print("Recall   :", recall_score(y_test, y_pred_xgb))
print("F1 Score :", f1_score(y_test, y_pred_xgb))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob_xgb))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb))

Accuracy : 0.7621621621621621
Precision: 0.8114754098360656
Recall   : 0.825
F1 Score : 0.8181818181818182
ROC-AUC  : 0.852051282051282

Confusion Matrix:
[[42 23]
 [21 99]]


In [42]:
# ==========================================
# SAVE AI STARTUP VALIDATOR MODEL
# ==========================================

import joblib
import os

# Create folder for saved models
os.makedirs("models", exist_ok=True)

# Save the trained Random Forest model
joblib.dump(
    best_rf,
    "models/startup_validator_random_forest.pkl"
)

# Save the preprocessing pipeline
joblib.dump(
    preprocessor,
    "models/startup_validator_preprocessor.pkl"
)

print("Models saved successfully!")
print("Random Forest : models/startup_validator_random_forest.pkl")
print("Preprocessor  : models/startup_validator_preprocessor.pkl")

Models saved successfully!
Random Forest : models/startup_validator_random_forest.pkl
Preprocessor  : models/startup_validator_preprocessor.pkl
